In [1]:
import os
import requests
import cdsapi
import platform
from datetime import datetime, timedelta

Yang dilakukan : 
1. Menghubungkan sistem dengan API eksternal (NASA Earthdata, FIRMS, GPM IMERG)
2. Menangani autentikasi login (biasanya menggunakan file .netrc)
3. Mengunduh file mentah data satelit (format .hdf, .nc, atau .tif) untuk rentang waktu sejarah yang ditentukan (misalnya 5 tahun ke belakang)
4. Menyimpan data mentah tersebut ke dalam penyimpanan lokal agar siap diproses

In [ ]:
NASA_USER = "ISI USER"
NASA_PASS = "ISI PASSWORD"

START = "2023-01-01"
END   = "2023-12-31"
BBOX = [108.5, -3.5, 114.0, 2.5]
TARGET_TILE = "h29v09"

dirs = ["../data/raw/rainfall", "../data/raw/lst", "../data/raw/vegetation", "../data/raw/wind_humidity", "../data/raw/aod"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

In [ ]:
def setup_auth_environment(username, password):
    if username == "USERNAME_EARTHDATA_ANDA":
        print("⚠️ PERINGATAN: Anda belum mengisi NASA_USER & NASA_PASS di bagian atas kode!")
        return False

    filename = "_netrc" if platform.system() == "Windows" else ".netrc"
    home_dir = os.path.expanduser("~")
    netrc_path = os.path.join(home_dir, filename)

    lines = [
        "machine urs.earthdata.nasa.gov login " + username + " password " + password,
        "machine ladsweb.modaps.eosdis.nasa.gov login " + username + " password " + password,
        "machine gpm1.gesdisc.eosdis.nasa.gov login " + username + " password " + password
    ]

    content = "\n".join(lines)
    
    try:
        with open(netrc_path, "w") as f:
            f.write(content) 
        if platform.system() != "Windows":
            os.chmod(netrc_path, 0o600)
        print(f"✅ Auth Berhasil: File {filename} telah dibuat/diupdate otomatis.")
        return True
    except Exception as e:
        print(f"❌ Gagal membuat file auth: {e}")
        return False

if not setup_auth_environment(NASA_USER, NASA_PASS):
    raise ValueError("Harap isi username & password NASA Earthdata Anda di bagian atas kode.")

session = requests.Session()

def get_nasa_filename(product, collection, year, doy):
    base_url = "https://ladsweb.modaps.eosdis.nasa.gov/archive/allData"
    url = f"{base_url}/{collection}/{product}/{year}/{doy:03d}.json"
    
    try:
        r = session.get(url) 
        
        if r.status_code == 200:
            data = r.json()
            
            file_list = []
            if isinstance(data, dict) and 'content' in data:
                file_list = data['content']
            elif isinstance(data, list):
                file_list = data
            
            for item in file_list:
                filename = item.get('name', '')
                
                if not filename.endswith(('.hdf', '.nc')):
                    continue
                
                if product.startswith("MOD"):
                    if TARGET_TILE in filename:
                        return filename
                else:
                    return filename
            
    except Exception as e:
        pass
        
    return None

✅ Auth Berhasil: File _netrc telah dibuat/diupdate otomatis.


In [ ]:
def download_gpm(date):
    yyyy, mm, dd = date[:4], date[5:7], date[8:10]
    url = f"https://gpm1.gesdisc.eosdis.nasa.gov/data/GPM_L3/GPM_3IMERGDF.07/{yyyy}/{mm}/3B-DAY.MS.MRG.3IMERG.{yyyy}{mm}{dd}-S000000-E235959.V07B.nc4"
    out = f"../data/raw/rainfall/gpm_{date}.nc4"

    if os.path.exists(out): return
    
    try:
        r = session.get(url, stream=True)
        if r.status_code == 200:
            with open(out, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024): f.write(chunk)
            print(f"GPM OK: {date}")
        else:
            # Fallback V07A
            r2 = session.get(url.replace("V07B", "V07A"), stream=True)
            if r2.status_code == 200:
                with open(out, 'wb') as f:
                    for chunk in r2.iter_content(chunk_size=1024): f.write(chunk)
                print(f"GPM OK (V07A): {date}")
            else: print(f"GPM FAIL: {r.status_code}")
    except Exception as e: print(f"GPM ERR: {e}")

In [ ]:
def download_modis_lst(date):
    yyyy = date[:4]
    doy = datetime.strptime(date, "%Y-%m-%d").timetuple().tm_yday
    fname = get_nasa_filename("MOD11A1", "61", yyyy, doy)

    if not fname: return

    url = f"https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/61/MOD11A1/{yyyy}/{doy:03d}/{fname}"
    out = f"../data/raw/lst/{fname}"

    if os.path.exists(out): return
    
    try:
        r = session.get(url, stream=True)
        if r.status_code == 200:
            with open(out, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024): f.write(chunk)
            print(f"MODIS LST OK: {date}")
        else: print(f"MODIS LST FAIL: {r.status_code}")
    except Exception as e: print(f"MODIS ERR: {e}")

In [ ]:
def download_modis_veg(date):
    yyyy = date[:4]
    doy = datetime.strptime(date, "%Y-%m-%d").timetuple().tm_yday
    fname = get_nasa_filename("MOD13A1", "61", yyyy, doy)

    if not fname: return 

    url = f"https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/61/MOD13A1/{yyyy}/{doy:03d}/{fname}"
    out = f"../data/raw/vegetation/{fname}"

    if os.path.exists(out): return
    
    try:
        r = session.get(url, stream=True)
        if r.status_code == 200:
            with open(out, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024): f.write(chunk)
            print(f"MODIS VEG OK: {date}")
        else: print(f"MODIS VEG FAIL: {r.status_code}")
    except Exception as e: print(f"VEG ERR: {e}")

In [ ]:
def download_viirs_aod(date):
    yyyy = date[:4]
    doy = datetime.strptime(date, "%Y-%m-%d").timetuple().tm_yday

    fname = get_nasa_filename("AERDB_D3_VIIRS_SNPP", "5200", yyyy, doy)
    if not fname: return
    url = f"https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/AERDB_D3_VIIRS_SNPP/{yyyy}/{doy:03d}/{fname}"
    out = f"../data/raw/aod/{fname}"
    if os.path.exists(out): return
    try:
        r = session.get(url, stream=True)
        if r.status_code == 200:
            with open(out, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024): f.write(chunk)
            print(f"VIIRS AOD OK: {date}")
        else: print(f"VIIRS AOD FAIL: {r.status_code}")
    except Exception as e: print(f"AOD ERR: {e}")

In [ ]:
def download_era5(date):

    c = cdsapi.Client()
    out = f"../data/raw/wind_humidity/era5_{date}.nc"
    if os.path.exists(out): return
    try:
        c.retrieve("reanalysis-era5-single-levels", {
            "product_type": "reanalysis",
            "variable": ["10m_u_component_of_wind", "10m_v_component_of_wind", 
                         "2m_dewpoint_temperature", "2m_temperature", "relative_humidity"],
            "year": date[:4], "month": date[5:7], "day": date[8:10],
            "time": [f"{h:02d}:00" for h in range(24)],
            "area": [BBOX[3], BBOX[0], BBOX[1], BBOX[2]],
            "format": "netcdf",
        }, out)
        print(f"ERA5 OK: {date}")
    except Exception as e: print(f"ERA5 ERR: {e}")

In [ ]:
if __name__ == "__main__":
    curr = datetime.strptime(START, "%Y-%m-%d")
    end  = datetime.strptime(END, "%Y-%m-%d")
    
    while curr <= end:
        d = curr.strftime("%Y-%m-%d")
        download_gpm(d)
        download_modis_lst(d)
        download_modis_veg(d)
        download_era5(d)
        download_viirs_aod(d)
        curr += timedelta(days=1)

2025-12-17 15:33:26,980 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-12-17 15:33:33,999 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-12-17 15:33:40,280 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-12-17 15:33:48,321 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-12-17 15:33:55,661 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-